# 演習2 解答編 ―― データ競合

> まず `ex02_race.ipynb` を自分で解いてから読んでください。

## 2-1 予測クイズの答え

2,000,000 にはなりません。毎回違う、それより小さい値になります。

`counter++` が「①読む ②足す ③書く」の3ステップに分かれており、
2つのスレッドがこれを同時に行うと、片方の書き込みがもう片方に**上書きされて消える**ためです。
どれだけ消えるかはタイミング次第なので、実行するたびに値が変わります。

## 発展課題1 の解答 ―― ループ回数を減らすと

次のセルで確かめてください。回数を変えて10回ずつ試し、正解なら ○、外れたら × を付けます。

In [ ]:
%%writefile ans02a.cpp
#include <iostream>
#include <thread>

long counter;
int N;

void add() { for (int i = 0; i < N; i++) counter++; }

void trial(int n, const char* label) {
    N = n;
    std::cout << label << " : ";
    for (int r = 0; r < 10; r++) {
        counter = 0;
        std::thread a(add), b(add);
        a.join(); b.join();
        std::cout << counter << (counter == 2L * N ? "○ " : "× ");
    }
    std::cout << "\n";
}

int main() {
    trial(1000,    "   1,000回 (期待値    2,000)");
    trial(1000000, "1,000,000回 (期待値 2,000,000)");
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans02a.cpp -o ans02a && ./ans02a

1,000回のほうは **ほとんど ○**（10回中7〜10回くらい）になったはずです。
1,000,000回のほうは **1回も当たりません**。
1,000 回程度に減らすと、**多くの場合ぴったり 2,000 になります**。
片方のスレッドがループを終えてから、もう片方が始まることが多いためです
（スレッドを起動する手間のほうが、ループ本体より重いくらいなので）。

しかし **たまに外します**。ここが肝心です。

ここから言えることは、競合バグの性質そのものです。

> **競合は「起きるか起きないか」ではなく「たまたま起きなかっただけ」。**

- テストでは通るのに、本番の負荷がかかると壊れる
- 手元では再現しないのに、別のマシンでは壊れる
- `cout` を1行足しただけで再現しなくなる（タイミングが変わるため）

「動いたから正しい」が通用しないのが並行プログラムの怖さで、
だからこそ「**共有するデータは必ず守る**」という規律が要ります。

## 発展課題2 の解答 ―― 競合するのはどれか

- `int index`（A だけが増やす） ⇒ **競合しない**
  　触るスレッドが1本だけ。共有変数でも、**使うのが1人なら安全**
- `bool stop`（A が書き、B が読む） ⇒ **競合する**
  　書く人と読む人がいる。`std::atomic<bool>` にするか、`mutex` で守る
- `Config conf`（起動前に設定、以後は読むだけ） ⇒ **競合しない**
  　スレッドを起動した時点で書き込みは終わっている。**全員が読むだけなら守らなくてよい**

ここから、守るべきかどうかの判断基準が引き出せます。

> **「共有しているから危ない」のではない。**
> **「複数のスレッドが触り、かつ少なくとも1つが書き込む」から危ない。**

つまり、次の**両方**が成り立つときだけ保護が要ります。

1. 2本以上のスレッドが同じデータを触る
2. そのうち少なくとも1本が**書き込む**

読むだけなら何人いても安全です（`Config conf` のケース）。
逆に、書く人が1人でも、読む人が別スレッドにいれば保護が必要です（`bool stop` のケース）。

`bool stop` のような停止フラグは「1バイトの読み書きだから大丈夫だろう」と
放置されがちですが、コンパイラの最適化によって
**ループの外に追い出され、永久に更新が見えなくなる**ことすらあります。
`std::atomic<bool>` を使うのが正解です。

## 発展課題3 の解答 ―― `taken[0]` と `taken[1]`

**競合していません。** 2つのスレッドは配列の**別々の要素**に書いており、
同じメモリを取り合ってはいないからです。正しさの点では問題ありません。

> **発展**：ただし性能上は不利になることがあります。
> CPU はメモリを「キャッシュライン」という 64 バイト程度のかたまりで扱うため、
> `taken[0]` と `taken[1]` が同じかたまりに入っていると、
> 片方を書き換えるたびにもう片方のキャッシュも無効化され、遅くなります。
> **フォルスシェアリング（false sharing）** と呼ばれる現象です。
> 次のセルで実際に測れます（正しさの問題ではないので、余力があれば見てください）。

In [ ]:
%%writefile ans02b.cpp
#include <iostream>
#include <thread>
#include <chrono>
using namespace std::chrono;

struct Near { long a; long b; };                  // 隣どうし（同じ64Bのかたまりに乗りやすい）
struct Far  { long a; char pad[128]; long b; };   // 128バイト離してある
Near nr;
Far  fr;

template<class F> long bench(F f) {
    auto t0 = steady_clock::now();
    std::thread t1(f, 0), t2(f, 1);
    t1.join(); t2.join();
    return duration_cast<milliseconds>(steady_clock::now() - t0).count();
}

int main() {
    auto near = [](int id) { for (long i = 0; i < 100000000; i++) (id ? nr.b : nr.a)++; };
    auto far  = [](int id) { for (long i = 0; i < 100000000; i++) (id ? fr.b : fr.a)++; };
    std::cout << "隣どうし   : " << bench(near) << " ms\n";
    std::cout << "128B離す   : " << bench(far)  << " ms\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans02b.cpp -o ans02b && ./ans02b

どちらも**別々の変数**を書き換えているだけで、競合はしていません。
それでも「隣どうし」のほうが 1.5〜2倍遅くなります。
CPU は 64 バイト程度のかたまり単位でメモリを扱うため、
片方を書き換えるともう片方のキャッシュも無効化されてしまうからです。

## 発展課題4 の解答 ―― ロックを取らない `size()`

```cpp
int size() { return queue_.size(); }     // ← ロックしていない
```

**これは競合しています。** 他のスレッドが `push` / `pop` している最中に
中身を読むことになるためです。

実際の困り方は2段階あります。

1. **返ってきた値がすでに古い。** `size()` が返った次の瞬間には、
   他のスレッドが `push` して値が変わっているかもしれません。
   したがって `if (q.size() < 10) q.push(x);` のような使い方は**必ず壊れます**
   （演習2-3 の「確かめてから取り出す」と同じ罠）。

2. **読んでいる最中に内部構造が書き換わる。** `std::queue` の内部は複数の変数でできており、
   その更新途中の状態を読むと、ありえない値が返ることがあります。

デバッグ表示にしか使わないなら実害は出にくいのですが、
**表示や計測のためのコードが競合を持ち込む**というのは、並行プログラムでよくある話です。
「動きを見るために足した1行」がバグの原因になることがある、と覚えておいてください。